## 1. Import lib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from scipy import sparse
from scipy.sparse import hstack
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score
from sklearn.feature_selection import chi2, f_classif, SelectFpr, SelectKBest
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
# from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

## 2. Load dataset

In [ ]:
df = pd.read_csv('/content/drive/Shareddrives/Học máy trong kinh doanh/Data/DataCoSupplyChainDataset.csv',encoding='latin-1')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Train model

In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 5.5 MB/s eta 0:00:00


### Baseline model

In [ ]:
seed = 2023
models = [
    DecisionTreeClassifier(random_state=seed),
    RandomForestClassifier(random_state=seed),
    LogisticRegression(random_state=seed),
    KNeighborsClassifier(),
    CatBoostClassifier(verbose=0, random_state=seed),
    MLPClassifier(random_state=seed),
    ExtraTreesClassifier(random_state=seed)
]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

def evaluate_models(models, X_train, y_train, X_test, y_test):
    results = []

    for model in models:
        model_name = model.__class__.__name__
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Tính cả 4 chỉ số
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        results.append((model_name, acc, prec, rec, f1))

    # Tạo DataFrame
    df_results = pd.DataFrame(results, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1'])
    # Sắp xếp theo F1 (hoặc chỉ số bạn quan tâm)
    df_results.sort_values(by='F1', ascending=False, inplace=True)
    return df_results

### Kết quả baseline

In [ ]:
df = evaluate_models(models, X_train_final, y_train, X_test_final, y_test)
print(df)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


                    Model  Accuracy  Precision    Recall        F1
6    ExtraTreesClassifier  0.891646   0.918754  0.880234  0.899082
0  DecisionTreeClassifier  0.875000   0.890695  0.880032  0.885332
5           MLPClassifier  0.866248   0.881176  0.873920  0.877533
1  RandomForestClassifier  0.841541   0.920029  0.778704  0.843488
3    KNeighborsClassifier  0.762852   0.781227  0.788251  0.784723
2      LogisticRegression  0.732606   0.806491  0.674092  0.734372
4      CatBoostClassifier  0.746565   0.875609  0.626863  0.730645


### Find best hyperparameter

In [ ]:
# Chọn cột
selected_categorical_features = ['Department Name', 'Category Name', 'Customer State',
                                 'Order Status', 'Order Country', 'Order Region',
                                 'Order State', 'Type', 'Customer City', 'Order City',
                                 'Shipping Mode']

selected_numerical_features = ['Days for shipment (scheduled)',
                               'Benefit per order', 'Sales per customer',
                               'Latitude', 'Longitude', 'Order Item Discount Rate',
                               'Order Item Product Price', 'Order Item Profit Ratio',
                               'Order Item Quantity']

X = df_selected[selected_categorical_features + selected_numerical_features]
y = df_selected['Late_delivery_risk']

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Encode categorical
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
encoder.fit(X_train[selected_categorical_features])

X_train_cate = encoder.transform(X_train[selected_categorical_features])
X_test_cate = encoder.transform(X_test[selected_categorical_features])

# Sparse matrix cho numerical
X_train_num = csr_matrix(X_train[selected_numerical_features].values)
X_test_num = csr_matrix(X_test[selected_numerical_features].values)

# Kết hợp
X_train_all = hstack((X_train_num, X_train_cate))
X_test_all = hstack((X_test_num, X_test_cate))

# Scale
scaler = StandardScaler(with_mean=False)
X_train_final = scaler.fit_transform(X_train_all)
X_test_final = scaler.transform(X_test_all)

# --- Dictionary các mô hình + Grid tham số ---
models_params = {
    'DecisionTree': {
        'model': DecisionTreeClassifier(random_state=42),
        'params': {
            'max_depth': [5, 10, 20, None],
            'min_samples_split': [2, 5, 10]
        }
    },
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [10, 20, None],
            'min_samples_split': [2, 5, 10]
        }
    },
    'ExtraTrees': {
        'model': ExtraTreesClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [10, 20, None],
            'min_samples_split': [2, 5, 10]
        }
    }
}


results_list = []
results_df = pd.DataFrame()

for name, mp in models_params.items():
    grid = GridSearchCV(mp['model'], mp['params'], cv=3, scoring='f1', n_jobs=-1)
    grid.fit(X_train_final, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test_final)

    result = {
        'Classifier': name,
        'Best_Params': grid.best_params_,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    }

    print(result)
    results_df = pd.concat([results_df, pd.DataFrame([result])], ignore_index=True)
    print(results_df)

{'Classifier': 'DecisionTree', 'Best_Params': {'max_depth': None, 'min_samples_split': 2}, 'Accuracy': 0.8709518710386942, 'Precision': 0.8931317273193234, 'Recall': 0.8800949542906208, 'F1-Score': 0.8865654175888473}
     Classifier                                  Best_Params  Accuracy  \
0  DecisionTree  {'max_depth': None, 'min_samples_split': 2}  0.870952   

   Precision    Recall  F1-Score  
0   0.893132  0.880095  0.886565  
{'Classifier': 'RandomForest', 'Best_Params': {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}, 'Accuracy': 0.822417735073655, 'Precision': 0.9116601385959626, 'Recall': 0.7641295014899743, 'F1-Score': 0.831400780348409}
     Classifier                                        Best_Params  Accuracy  \
0  DecisionTree        {'max_depth': None, 'min_samples_split': 2}  0.870952   
1  RandomForest  {'max_depth': None, 'min_samples_split': 2, 'n...  0.822418   

   Precision    Recall  F1-Score  
0   0.893132  0.880095  0.886565  
1   0.911660  0

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


{'Classifier': 'ExtraTrees', 'Best_Params': {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}, 'Accuracy': 0.8842358116516655, 'Precision': 0.9149986866298923, 'Recall': 0.8796908934794686, 'F1-Score': 0.8969974764381727}
     Classifier                                        Best_Params  Accuracy  \
0  DecisionTree        {'max_depth': None, 'min_samples_split': 2}  0.870952   
1  RandomForest  {'max_depth': None, 'min_samples_split': 2, 'n...  0.822418   
2    ExtraTrees  {'max_depth': None, 'min_samples_split': 2, 'n...  0.884236   

   Precision    Recall  F1-Score  
0   0.893132  0.880095  0.886565  
1   0.911660  0.764130  0.831401  
2   0.914999  0.879691  0.896997  


## Pickle

#### DT

In [ ]:
import pandas as pd
import pickle
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler, AllKNN
from imblearn.combine import SMOTETomek
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.sparse import csr_matrix, hstack

# CHUẨN BỊ DỮ LIỆU
selected_categorical_features = [
    'Department Name', 'Category Name', 'Customer State',
    'Order Status', 'Order Country', 'Order Region',
    'Order State', 'Type', 'Customer City', 'Order City',
    'Shipping Mode'
]

selected_numerical_features = [
    'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer',
    'Latitude', 'Longitude', 'Order Item Discount Rate',
    'Order Item Product Price', 'Order Item Profit Ratio',
    'Order Item Quantity'
]

X = df_selected[selected_categorical_features + selected_numerical_features]
y = df_selected['Late_delivery_risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Encode categorical
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
encoder.fit(X_train[selected_categorical_features])
X_train_cate = encoder.transform(X_train[selected_categorical_features])
X_test_cate = encoder.transform(X_test[selected_categorical_features])

# Sparse matrix cho numerical
X_train_num = csr_matrix(X_train[selected_numerical_features].values)
X_test_num = csr_matrix(X_test[selected_numerical_features].values)

# Gộp lại
X_train_all = hstack((X_train_num, X_train_cate))
X_test_all = hstack((X_test_num, X_test_cate))

# Scale
scaler = StandardScaler(with_mean=False)
X_train_final = scaler.fit_transform(X_train_all)
X_test_final = scaler.transform(X_test_all)

# ================== HÀM ĐÁNH GIÁ ==================
def evaluate_resampling(model, resamplers, X_train, y_train, X_test, y_test):
    results = []
    trained_models = {}
    for name, sampler in resamplers.items():
        if sampler is None:
            X_res, y_res = X_train, y_train
        else:
            X_res, y_res = sampler.fit_resample(X_train, y_train)

        # clone model
        model_clone = model.__class__(**model.get_params())
        model_clone.fit(X_res, y_res)
        y_pred = model_clone.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        results.append((name, acc, prec, rec, f1))
        trained_models[name] = model_clone

    df_results = pd.DataFrame(results, columns=['Technique', 'Accuracy', 'Precision', 'Recall', 'F1'])
    df_results.sort_values(by='F1', ascending=False, inplace=True)
    return df_results, trained_models

# ================== RESAMPLERS ==================
resamplers = {
    "Without": None,
    "RUS": RandomUnderSampler(random_state=42),
    "ROS": RandomOverSampler(random_state=42),
    "SMOTE": SMOTE(random_state=42),
    "SMOTETomek": SMOTETomek(random_state=42),
    "AllKNN": AllKNN()
}

# ================== CHỈ DÙNG DECISION TREE ==================
decision_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42
)

df, trained = evaluate_resampling(decision_tree, resamplers, X_train_final, y_train, X_test_final, y_test)
print("\n=== DecisionTree ===")
print(df.to_string(index=False))

# Package
package = {
    "encoder": encoder,
    "scaler": scaler,
    "results": {"DecisionTree": df},
    "models": {"DecisionTree": trained}
}

with open(r"D:\decisiontree_resampling.pkl", "wb") as f:
    pickle.dump(package, f)

print("Đã lưu encoder, scaler, kết quả và toàn bộ DecisionTree vào D:\\decisiontree_resampling.pkl")



=== DecisionTree ===
 Technique  Accuracy  Precision   Recall       F1
   Without  0.885799   0.886318 0.885799 0.885962
       ROS  0.885828   0.886001 0.885828 0.885898
       RUS  0.858768   0.862274 0.858768 0.859344
    AllKNN  0.824820   0.845437 0.824820 0.825546
     SMOTE  0.728070   0.730267 0.728070 0.728820
SMOTETomek  0.711371   0.714006 0.711371 0.712252
Đã lưu encoder, scaler, kết quả và toàn bộ DecisionTree vào D:\decisiontree_resampling.pkl


In [ ]:
with open("/content/decisiontree_resampling.pkl", "wb") as f:
    pickle.dump(package, f)

In [ ]:
from google.colab import files
files.download("/content/decisiontree_resampling.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

###RF

In [ ]:
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler, AllKNN
from imblearn.combine import SMOTETomek
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.sparse import csr_matrix, hstack

# CHUẨN BỊ DỮ LIỆU
selected_categorical_features = [
    'Department Name', 'Category Name', 'Customer State',
    'Order Status', 'Order Country', 'Order Region',
    'Order State', 'Type', 'Customer City', 'Order City',
    'Shipping Mode'
]

selected_numerical_features = [
    'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer',
    'Latitude', 'Longitude', 'Order Item Discount Rate',
    'Order Item Product Price', 'Order Item Profit Ratio',
    'Order Item Quantity'
]

X = df_selected[selected_categorical_features + selected_numerical_features]
y = df_selected['Late_delivery_risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Encode categorical
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
encoder.fit(X_train[selected_categorical_features])
X_train_cate = encoder.transform(X_train[selected_categorical_features])
X_test_cate = encoder.transform(X_test[selected_categorical_features])

# Sparse matrix cho numerical
X_train_num = csr_matrix(X_train[selected_numerical_features].values)
X_test_num = csr_matrix(X_test[selected_numerical_features].values)

# Gộp lại
X_train_all = hstack((X_train_num, X_train_cate))
X_test_all = hstack((X_test_num, X_test_cate))

# Scale
scaler = StandardScaler(with_mean=False)
X_train_final = scaler.fit_transform(X_train_all)
X_test_final = scaler.transform(X_test_all)

# ================== HÀM ĐÁNH GIÁ ==================
def evaluate_resampling(model, resamplers, X_train, y_train, X_test, y_test):
    results = []
    trained_models = {}
    for name, sampler in resamplers.items():
        if sampler is None:
            X_res, y_res = X_train, y_train
        else:
            X_res, y_res = sampler.fit_resample(X_train, y_train)

        # clone model
        model_clone = model.__class__(**model.get_params())
        model_clone.fit(X_res, y_res)
        y_pred = model_clone.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        results.append((name, acc, prec, rec, f1))
        trained_models[name] = model_clone

    df_results = pd.DataFrame(results, columns=['Technique', 'Accuracy', 'Precision', 'Recall', 'F1'])
    df_results.sort_values(by='F1', ascending=False, inplace=True)
    return df_results, trained_models

# ================== RESAMPLERS ==================
resamplers = {
    "Without": None,
    "RUS": RandomUnderSampler(random_state=42),
    "ROS": RandomOverSampler(random_state=42),
    "SMOTE": SMOTE(random_state=42),
    "SMOTETomek": SMOTETomek(random_state=42),
    "AllKNN": AllKNN()
}

# ================== CHỈ DÙNG RANDOM FOREST ==================
random_forest = RandomForestClassifier(
    criterion="gini",
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

df, trained = evaluate_resampling(random_forest, resamplers, X_train_final, y_train, X_test_final, y_test)
print("\n=== RandomForest ===")
print(df.to_string(index=False))

# Package
package = {
    "encoder": encoder,
    "scaler": scaler,
    "results": {"RandomForest": df},
    "models": {"RandomForest": trained}
}

with open(r"D:\randomforest_resampling.pkl", "wb") as f:
    pickle.dump(package, f)

print("Đã lưu encoder, scaler, kết quả và toàn bộ RandomForest vào D:\\randomforest_resampling.pkl")



=== RandomForest ===
 Technique  Accuracy  Precision  Recall     F1
   Without    0.8278     0.8413  0.8278 0.8287
       ROS    0.8216     0.8412  0.8216 0.8224
    AllKNN    0.7945     0.8352  0.7945 0.7941
       RUS    0.7779     0.8168  0.7779 0.7775
     SMOTE    0.7457     0.7746  0.7457 0.7460
SMOTETomek    0.7436     0.7728  0.7436 0.7439


### ET

In [ ]:
import pandas as pd
import pickle
from sklearn.ensemble import ExtraTreesClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler, AllKNN
from imblearn.combine import SMOTETomek
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.sparse import csr_matrix, hstack

# ================== CHUẨN BỊ DỮ LIỆU ==================
selected_categorical_features = [
    'Department Name', 'Category Name', 'Customer State',
    'Order Status', 'Order Country', 'Order Region',
    'Order State', 'Type', 'Customer City', 'Order City',
    'Shipping Mode'
]

selected_numerical_features = [
    'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer',
    'Latitude', 'Longitude', 'Order Item Discount Rate',
    'Order Item Product Price', 'Order Item Profit Ratio',
    'Order Item Quantity'
]

# Lấy dữ liệu
X = df_selected[selected_categorical_features + selected_numerical_features]
y = df_selected['Late_delivery_risk']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Encode categorical
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
encoder.fit(X_train[selected_categorical_features])
X_train_cate = encoder.transform(X_train[selected_categorical_features])
X_test_cate = encoder.transform(X_test[selected_categorical_features])

# Sparse matrix cho numerical (chuyển sang float32 để nhẹ hơn)
X_train_num = csr_matrix(X_train[selected_numerical_features].astype('float32').values)
X_test_num = csr_matrix(X_test[selected_numerical_features].astype('float32').values)

# Gộp lại
X_train_all = hstack((X_train_num, X_train_cate))
X_test_all = hstack((X_test_num, X_test_cate))

# Scale
scaler = StandardScaler(with_mean=False)
X_train_final = scaler.fit_transform(X_train_all)
X_test_final = scaler.transform(X_test_all)

# ================== HÀM ĐÁNH GIÁ ==================
def evaluate_resampling(model, resamplers, X_train, y_train, X_test, y_test):
    results = []
    trained_models = {}

    for name, sampler in resamplers.items():
        print(f"\n🚀 Training with {name} ...")
        if sampler is None:
            X_res, y_res = X_train, y_train
        else:
            X_res, y_res = sampler.fit_resample(X_train, y_train)

        # clone model
        model_clone = model.__class__(**model.get_params())
        model_clone.fit(X_res, y_res)
        y_pred = model_clone.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        results.append((name, acc, prec, rec, f1))
        trained_models[name] = model_clone

    df_results = pd.DataFrame(results, columns=['Technique', 'Accuracy', 'Precision', 'Recall', 'F1'])
    df_results.sort_values(by='F1', ascending=False, inplace=True)
    return df_results, trained_models

# ================== RESAMPLERS ==================
resamplers = {
    "Without": None,
    "RUS": RandomUnderSampler(random_state=42),
    "ROS": RandomOverSampler(random_state=42),
    "SMOTE": SMOTE(random_state=42),
    "SMOTETomek": SMOTETomek(random_state=42),
    "AllKNN": AllKNN()
}

# ================== EXTRA TREES ==================
extra_trees = ExtraTreesClassifier( n_estimators=100, max_depth=None, min_samples_split=2, min_samples_leaf=1, class_weight="balanced", random_state=42, n_jobs=-1 )

df, trained = evaluate_resampling(extra_trees, resamplers, X_train_final, y_train, X_test_final, y_test)
print("\n=== ExtraTrees ===")
print(df.to_string(index=False))

# ================== LƯU PACKAGE ==================
package = {
    "encoder": encoder,
    "scaler": scaler,
    "results": {"ExtraTrees": df},
    "models": {"ExtraTrees": trained}
}

with open(r"D:\extratrees_resampling.pkl", "wb") as f:
    pickle.dump(package, f)

print("✅ Đã lưu encoder, scaler, kết quả và toàn bộ ExtraTrees vào D:\\extratrees_resampling.pkl")


🚀 Training with Without ...

🚀 Training with RUS ...

🚀 Training with ROS ...

🚀 Training with SMOTE ...

🚀 Training with SMOTETomek ...

🚀 Training with AllKNN ...

=== ExtraTrees ===
 Technique  Accuracy  Precision   Recall       F1
   Without  0.883860   0.885624 0.883860 0.884210
       ROS  0.882326   0.883988 0.882326 0.882669
       RUS  0.862270   0.871757 0.862270 0.863005
    AllKNN  0.831332   0.861770 0.831332 0.831623
     SMOTE  0.796284   0.808839 0.796284 0.797362
SMOTETomek  0.792435   0.805456 0.792435 0.793526
✅ Đã lưu encoder, scaler, kết quả và toàn bộ ExtraTrees vào D:\extratrees_resampling.pkl


In [ ]:
with open("/content/extratrees_resampling.pkl", "wb") as f:
    pickle.dump(package, f)


In [ ]:
!cp /content/extratrees_resampling.pkl /content/drive/MyDrive/

In [ ]:
!find /content/drive -name "extratrees_resampling.pkl"

/content/drive/MyDrive/extratrees_resampling.pkl


## Load model pkl

In [ ]:
import os
import pickle
import pandas as pd
import gc

model_files = {
    "DecisionTree": "/content/drive/Shareddrives/Học máy trong kinh doanh/MODEL/RESULT_PHASE1/decisiontree_resampling.pkl",
    "ExtraTrees": "/content/drive/Shareddrives/Học máy trong kinh doanh/MODEL/RESULT_PHASE1/extratrees_resampling.pkl",
    "RandomForest": "/content/drive/Shareddrives/Học máy trong kinh doanh/MODEL/RESULT_PHASE1/randomforest_resampling.pkl"
}

models_ros = {}

for name, path in model_files.items():
    print(f"Đang load {name} từ {path} ...")

    if not os.path.exists(path):
        print(f"Không tìm thấy file: {path}")
        continue

    try:
        with open(path, "rb") as f:
            pkg = pickle.load(f)
        print(f"File {name} load thành công")

        # Kiểm tra cấu trúc và lấy encoder, scaler, model
        encoder = pkg.get("encoder")
        scaler = pkg.get("scaler")
        model = pkg.get("models", {}).get(name, {}).get("ROS")

        if encoder is None or scaler is None or model is None:
            print(f"Thiếu encoder/scaler/model trong {name}, bỏ qua.")
            continue

        models_ros[name] = {
            "encoder": encoder,
            "scaler": scaler,
            "model": model
        }
        print(f"Loaded {name} ROS hoàn chỉnh")

    except (EOFError, AttributeError, MemoryError, pickle.UnpicklingError) as e:
        print(f"Lỗi khi load {name}: {type(e).__name__} - {e}")
        continue
    except Exception as e:
        print(f"Lỗi không xác định khi load {name}: {e}")
        continue

    del pkg
    gc.collect()

Đang load DecisionTree từ /content/drive/Shareddrives/Học máy trong kinh doanh/MODEL/RESULT_PHASE1/decisiontree_resampling.pkl ...
File DecisionTree load thành công
Loaded DecisionTree ROS hoàn chỉnh
Đang load ExtraTrees từ /content/drive/Shareddrives/Học máy trong kinh doanh/MODEL/RESULT_PHASE1/extratrees_resampling.pkl ...
File ExtraTrees load thành công
Loaded ExtraTrees ROS hoàn chỉnh
Đang load RandomForest từ /content/drive/Shareddrives/Học máy trong kinh doanh/MODEL/RESULT_PHASE1/randomforest_resampling.pkl ...
File RandomForest load thành công
Loaded RandomForest ROS hoàn chỉnh


In [ ]:
def normalize_input(input_dict):
    cleaned = {}
    for k, v in input_dict.items():
        if isinstance(v, str):
            cleaned[k] = v.strip().title()  # bỏ khoảng trắng + viết hoa đầu từ
        else:
            cleaned[k] = v
    return cleaned

In [ ]:
def predict_one_order_fast_only_active(model_package, input_dict):
    input_dict = normalize_input(input_dict)
    df = pd.DataFrame([input_dict])

    encoder = model_package["encoder"]
    model = model_package["model"]

    # Chuẩn bị dữ liệu
    X_num = df[selected_numerical_features_trong].values
    X_cate = encoder.transform(df[selected_categorical_features_trong])
    X_all = hstack((X_num, X_cate))

    # 2. Dự đoán
    pred = model.predict(X_all)[0]
    prob = model.predict_proba(X_all)[0]
    label = "Trễ" if pred == 1 else "Không trễ"

    print(f" Dự đoán: {label}")
    print(f" Xác suất Không trễ: {prob[0]:.4f} | Trễ: {prob[1]:.4f}")

    # Tính toán feature importance
    importances = model.feature_importances_

    #Numerical features
    features = list(selected_numerical_features_trong)
    values = [input_dict[f] for f in selected_numerical_features_trong]
    imps = importances[:len(selected_numerical_features_trong)].tolist()

    #  Categorical features: chỉ giá trị đang active
    start_idx = len(selected_numerical_features_trong)
    for i, cat in enumerate(selected_categorical_features_trong):
        cat_columns = encoder.categories_[i]
        val = input_dict[cat]
        try:
            idx_in_cat = list(cat_columns).index(val)
        except ValueError:
            continue  # nếu value không có trong encoder

        # Vị trí trong toàn bộ vector one-hot
        feature_idx = start_idx + sum(len(c) for c in encoder.categories_[:i]) + idx_in_cat
        imp_value = importances[feature_idx]

        features.append(cat)
        values.append(val)
        imps.append(imp_value)

    #  4. Gom kết quả gọn
    df_feat = (
        pd.DataFrame({"Feature": features, "Value": values, "Importance": imps})
        .sort_values("Importance", ascending=False)
        .reset_index(drop=True)
    )

    print("\n📋 Các feature ảnh hưởng mạnh nhất trong đơn hàng này:")
    print(df_feat.head(15))

    return label, prob, df_feat

In [ ]:
selected_categorical_features_trong = ['Department Name', 'Category Name', 'Customer State',
                                 'Order Status', 'Order Country', 'Order Region',
                                 'Order State', 'Type', 'Customer City', 'Order City',
                                 'Shipping Mode']
selected_numerical_features_trong = ['Days for shipment (scheduled)',
       'Benefit per order', 'Sales per customer',
       'Latitude', 'Longitude', 'Order Item Discount Rate',
       'Order Item Product Price', 'Order Item Profit Ratio',
       'Order Item Quantity']

## Load Model + Evaluate

In [ ]:
import os
import pickle
import pandas as pd
import gc

model_files = {
    "DecisionTree": "/content/drive/Shareddrives/Học máy trong kinh doanh/MODEL/RESULT_PHASE1/decisiontree_resampling.pkl",
    "ExtraTrees": "/content/drive/Shareddrives/Học máy trong kinh doanh/MODEL/RESULT_PHASE1/extratrees_resampling.pkl",
    "RandomForest": "/content/drive/Shareddrives/Học máy trong kinh doanh/MODEL/RESULT_PHASE1/randomforest_resampling.pkl"
}

results_data = {}

for name, path in model_files.items():
    print(f"\nĐang load bảng kết quả từ {name} ...")

    if not os.path.exists(path):
        print(f"Không tìm thấy file: {path}")
        continue

    try:
        with open(path, "rb") as f:
            pkg = pickle.load(f)

        if "results" not in pkg:
            print(f"Không có 'results' trong {name}")
            continue

        results = pkg["results"]

        if isinstance(results, dict):
            inner_df = None
            for v in results.values():
                if isinstance(v, pd.DataFrame):
                    inner_df = v
                    break
            if inner_df is not None:
                results = inner_df
            else:
                results = pd.DataFrame([results])

        elif isinstance(results, list):
            results = pd.DataFrame(results)

        elif not isinstance(results, pd.DataFrame):
            print(f"Dữ liệu 'results' trong {name} không phải dạng DataFrame, dict hoặc list.")
            continue

        results_data[name] = results
        print(f"Đã load bảng kết quả của {name} ({len(results)} dòng)")

    except Exception as e:
        print(f"Lỗi khi load {name}: {e}")

    finally:
        del pkg
        gc.collect()

print(f"\n Tổng số mô hình load thành công: {len(results_data)} / {len(model_files)}")
print(f" Các mô hình có sẵn: {list(results_data.keys())}")


Đang load bảng kết quả từ DecisionTree ...
Đã load bảng kết quả của DecisionTree (6 dòng)

Đang load bảng kết quả từ ExtraTrees ...
Đã load bảng kết quả của ExtraTrees (6 dòng)

Đang load bảng kết quả từ RandomForest ...
Đã load bảng kết quả của RandomForest (6 dòng)

 Tổng số mô hình load thành công: 3 / 3
 Các mô hình có sẵn: ['DecisionTree', 'ExtraTrees', 'RandomForest']


In [ ]:
def get_model_metrics(model_name: str, technique: str):
    """
    Trả về Accuracy, Precision, Recall, F1 từ bảng kết quả đã load.
    """
    if model_name not in results_data:
        print(f"Mô hình '{model_name}' chưa được load.")
        print(f"Các mô hình sẵn sàng: {list(results_data.keys())}")
        return None

    df = results_data[model_name]

    if "Technique" not in df.columns:
        print(f"Không có cột 'Technique' trong bảng kết quả của {model_name}.")
        print(f"Các cột có sẵn: {list(df.columns)}")
        return None

    row = df[df["Technique"].str.lower() == technique.lower()]
    if row.empty:
        print(f"Không tìm thấy technique '{technique}' trong {model_name}.")
        print(f"Các technique có sẵn: {list(df['Technique'])}")
        return None

    metrics = row.iloc[0][["Accuracy", "Precision", "Recall", "F1"]].to_dict()

    print(f"\nKết quả cho {model_name} - {technique}:")
    for k, v in metrics.items():
        print(f"   {k}: {v:.6f}")
    return metrics

In [ ]:
metrics = get_model_metrics("ExtraTrees", "SMOTETomek")



✅ Kết quả cho ExtraTrees - SMOTETomek:
   Accuracy: 0.792435
   Precision: 0.805456
   Recall: 0.792435
   F1: 0.793526
